# Environment Setup & Prerequisites

Olu’s first task at AfyaPlus is to prepare the tools she’ll use throughout the programme. Use this checklist to set up and verify your development environment before starting the technical lessons.

## 1. Install the required tools

- [Python 3.10 or later](https://www.python.org/downloads/) — the programming language used in the labs.
- [Visual Studio Code](https://code.visualstudio.com/) — the primary editor for code-alongs, debugging, and file management.
- [Git](https://git-scm.com/downloads) — version control for tracking changes and submitting work.
- **pip** — Python’s package manager; included with Python. You can also use Poetry or UV to manage the packages used in the project. _I'm currently using UV_
- [Ollama](https://ollama.ai/download) — local inference engine used in Tuesday’s session.

Open a terminal and check your Python version:


In [41]:
!python --version

Python 3.14.6


On macOS or Linux, use `python3 --version` if needed. Confirm the version is **3.10 or later**.

In [42]:
!python3 --version

Python 3.14.6


## 2. Configure VS Code

Install these extensions:
- **Python** by Microsoft
- **Pylance**
- **Python Environment Manager**

| You can check this on the extensions manager on your VS Code Sidebar

## 3. Create the project and virtual environment

Open VS Code, then select **Terminal > New Terminal**. Run:
```bash
    # Create project directory
    mkdir afyaplus-ai-week1
    cd afyaplus-ai-week1

    # Create a virtual environment
    python -m venv venv

    # Activate the virtual environment
    # On macOS/Linux:
    source venv/bin/activate
    # On Windows:
    venv\Scripts\activate

    # Install required packages
    pip install openai httpx numpy python-dotenv
```

| You can also create these folders using the GUI (Graphic User Interface) depending on your OS (File Explorer, Finder, or whichever Linux GUI you may be using)


In [43]:
# I usually use UV for my package management, so the commands below may differ
PROJECT = "/Users/nelsonombuya/Documents/Projects/Moringa/Applied AI Engineering (Cohort 2)"
!mkdir -p "$PROJECT"
!cd "$PROJECT" && \
    ([ -f pyproject.toml ] || uv init --bare --no-readme --no-pin-python --vcs none --no-package)
!cd "$PROJECT" && uv venv .venv --allow-existing
!cd "$PROJECT" && uv sync

Using CPython 3.14.6
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 109 packages in 23ms
Checked 105 packages in 27ms


Activate the environment:

```bash
# macOS/Linux
source venv/bin/activate

# Windows
venv\Scripts\activate
```

In [44]:
# I'll use the PROJECT variable I declared above to activate the virtual environment.
!source "$PROJECT/.venv/bin/activate"

Install the required packages:

```bash
pip install openai httpx numpy python-dotenv
```

These packages provide an OpenAI SDK, an HTTP client, numerical arrays for Monday’s labs, and support for loading environment variables.

In [45]:
# Since I'll be using uv, I can use:
!uv add openai httpx numpy python-dotenv

Resolved 109 packages in 4ms
Checked 105 packages in 3ms


## 4. Check local-model hardware

Before Tuesday’s local-model session, confirm your computer has:

- **At least 5 GB** of free SSD storage
- **At least 8 GB** of system RAM

## 5. Store your API key securely

Create an OpenAI account or sign in at [platform.openai.com](https://platform.openai.com/), then create a secret API key from **API Keys** in your account settings.

From the project folder, create a `.env` file containing your key:
Replace `sk-your-key-here` with your actual key. Never hardcode API keys in source code or commit them to Git.

In [46]:
# Skipping this command since I already have my API key in my environment variables.
# If you don't have your API key in your environment variables, you can uncomment the line below and add your OpenAI API key to a .env file.
# !echo "OPENAI_API_KEY=sk-your-key-here" > .env

Create a `.gitignore` file so sensitive files and generated files are not tracked:

In [47]:
# Skipping this command since I already have a .gitignore file in my project directory.
# If you don't have a .gitignore file in your project directory, you can uncomment the line below to create one with some common patterns to ignore.
# !printf "%s\n" ".env" "venv/" "__pycache__/" "*.pyc" > .gitignore

## 6. Verify the setup
Run these commands in the VS Code terminal (_we'll use them to test out that our initial steps were set up correctly_):

In [48]:
!python --version
!python -c "import openai, httpx, numpy, dotenv; print('All packages installed')"
!python -c "from dotenv import load_dotenv; import os; load_dotenv(); k=os.getenv('OPENAI_API_KEY'); print('OpenAI API Key loaded' if k else 'OpenAI API Key MISSING')"
!python -c "from dotenv import load_dotenv; import os; load_dotenv(); k=os.getenv('OPENROUTER_API_KEY'); print('OpenRouter API Key loaded' if k else 'OpenRouter API Key MISSING')"

Python 3.14.6
All packages installed
OpenAI API Key loaded
OpenRouter API Key loaded


Resolve any errors before moving on. Keep `.env` private, and confirm that `.gitignore` excludes it.

---

## tokenisation_lab.py

In [4]:
import numpy as np

def simple_numpy_tokenizer(text):
    # 1. Basic cleaning and splitting
    # We convert to lowercase and remove simple punctuation
    clean_text = text.lower().replace(".", "").replace("?", "")
    tokens = clean_text.split()

    # 2. Convert to NumPy array
    token_array = np.array(tokens)

    # 3. Create a vocabulary (unique tokens)
    # This represents the internal 'dictionary' our system recognises
    vocab, inverse_indices = np.unique(token_array, return_inverse=True)

    return token_array, vocab, inverse_indices

# Real patient case from the AfyaPlus portal
patient_query = "My chest hurts. Is it a heart attack?"
tokens, vocabulary, token_ids = simple_numpy_tokenizer(patient_query)

print("--- Tokenisation Results ---")
print(f"Original Text: {patient_query}")
print(f"Tokens: {tokens}")
print(f"Vocabulary: {vocabulary}")
print(f"Token IDs (Numerical Representation): {token_ids}")

--- Tokenisation Results ---
Original Text: My chest hurts. Is it a heart attack?
Tokens: ['my' 'chest' 'hurts' 'is' 'it' 'a' 'heart' 'attack']
Vocabulary: ['a' 'attack' 'chest' 'heart' 'hurts' 'is' 'it' 'my']
Token IDs (Numerical Representation): [7 2 4 5 6 0 3 1]


## challenge1_tokeniser.py (starter)

In [3]:
import numpy as np

def emergency_tokeniser(text):
    """Split text into tokens while preserving '!' as separate tokens."""
    text = text.lower()
    # TODO 1: Pad '!' with spaces so split() treats them as separate tokens.
    text = text.replace('!', ' ! ')

    # TODO 2: Remove any other punctuation that should not be a token.
    text = text.replace('.', '').replace(',', '').replace('?', '').replace(';', '').replace(':', '').replace('"', '').replace("'", '')

    # TODO 3: Split on whitespace and return a NumPy array.
    tokens = np.array(text.split())  # replace this with the real result
    return tokens

# Test cases
samples = [
    "Help!!!",
    "I cannot breathe!",
    "My chest hurts.",
    "My chest hurts. Is it a heart attack?",
    "My chest hurts!!!",
]
for s in samples:
    print(s, "->", emergency_tokeniser(s))

# Expected for "Help!!!": ['help', '!', '!', '!']

Help!!! -> ['help' '!' '!' '!']
I cannot breathe! -> ['i' 'cannot' 'breathe' '!']
My chest hurts. -> ['my' 'chest' 'hurts']
My chest hurts. Is it a heart attack? -> ['my' 'chest' 'hurts' 'is' 'it' 'a' 'heart' 'attack']
My chest hurts!!! -> ['my' 'chest' 'hurts' '!' '!' '!']
